# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    get_feature_transforms, get_model_information
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"

## Extract Features **X** Used in Model

In [7]:
# create dict to store features
features = {}

In [8]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # create internal dict for analysis features
    features[i] = {}
    
    # get the features from each analysis
    # this should include the independent and control variables
    ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
    control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in ind_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    for dict_idx, var in enumerate(ind_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
        
    # save updated independent variables in features dict
    features[i]['independent_variables'] = ind_vars
    
    # tkae same approach for control variables
    for dict_idx, var in enumerate(control_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
    
    # save updated control variables in features dict
    features[i]['control_variables'] = control_vars

[2025-11-13 10:05:48.19][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-13 10:05:49.60][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [9]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Response *y* used in Model

In [10]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # get the features from each analysis
    # this should include the independent and control variables
    response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in response_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    # for dict_idx, var in enumerate(response_vars):
    transform_responses = get_feature_transforms(llm_assistant,
                                                 transform_code,
                                                 response_vars['columns'],
                                                 response_vars['description'])
    response_vars['transform_code'] = [response.text[0].content \
        for response in transform_responses]

    # save updated response variables in features dict
    features[i]['response_variables'] = response_vars

[2025-11-13 10:05:50.94][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-13 10:05:51.62][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [11]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Model Class Used

In [12]:
# create dict to store model information
model_info = {}

In [13]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):

    # get model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']

    # use helper function to get model information
    model_info[i] = get_model_information(llm_assistant, model_code)

In [14]:
# view model_info to ensure correctness
model_info

{0: '{\n  "model_library": "statsmodels (imported as statsmodels.api as sm)",\n  "model_class": "sm.GLM with family=sm.families.NegativeBinomial (fallback: sm.GLM with family=sm.families.Poisson)), and sm.OLS for the log-damage robustness check",\n  "model_parameters": "GLM NegativeBinomial: family=sm.families.NegativeBinomial() (uses default log link); fallback GLM Poisson: family=sm.families.Poisson() with fit(cov_type=\'HC0\') for robust SEs; OLS: sm.OLS(...).fit(cov_type=\'HC1\') for robust SEs. Design matrix: predictors = [\'masfem_std\',\'gender_mf\',\'wind\',\'category\',\'min\',\'year\',\'elapsedyrs\',\'masfem_x_category\']; constant added via sm.add_constant(X_model). Interaction created as masfem_x_category = masfem_std * (category - category.mean()).",\n  "model_formula_fitting_code": "try:\\n    nb_model = sm.GLM(y_counts, X_model, family=sm.families.NegativeBinomial()).fit()\\n    results[\'neg_bin_alldeaths\'] = nb_model\\nexcept Exception as e:\\n    pois = sm.GLM(y_coun

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [15]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [16]:
# run the transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    transformed_datasets[i] = transform_func(data.copy()) # use copy of dataset

# run the model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    model_results[i] = model_func(transformed_datasets[i].copy()) # use copy

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [17]:
# view the first model result as a sanity check
model_results[0]

{'neg_bin_alldeaths': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x7315c7890e80>,
 'ols_log_ndam15': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x73162c1f0790>}

In [18]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-13 10:06:08.94][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:06:57.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  48.44 seconds
[2025-11-13 10:06:57.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 10:06:57.41][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:07:39.83][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  42.42 seconds
[2025-11-13 10:07:39.83][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [19]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts statistics relating the effect of hurricane name femininity (\'masfem_std\')\n    (and as a secondary check \'gender_mf\') on death counts (neg_bin_alldeaths) and\n    on log damage (ols_log_ndam15).\n\n    Returns a dict with:\n      - "object": dict containing numeric summaries for each model and variable\n      - "description": short interpretation of whether results support the hypothesis\n                       that more-feminine hurricane names are associated with outcomes\n                       consistent with fewer precautionary measures (i.e., more deaths).\n    """\n    import numpy as np\n\n    def extract_from_result(res, var):\n        out = {"present": False}\n        if res is None:\n            return out\n        try:\n            params = getattr(res, "params", None)\n            if params is None or var not in params.index:\n                return out\n            coef = float(params[var])\n        

In [20]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [21]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    interpretation_code = final_answer_code[i]
    
    # get the interpretation output
    interpretation_output = final_answers[i]
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-13 10:07:40.07][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-13 10:07:47.47][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.40 seconds
[2025-11-13 10:07:47.47][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 10:07:47.49][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:07:54.14][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.65 seconds
[2025-11-13 10:07:54.15][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [22]:
conclusions

{0: '{\n  "answer": "No",\n  "justification": "The primary negative-binomial model yields a positive masfem_std coefficient (IRR≈1.25) but it is not statistically significant (p=0.555; CI includes 1). The OLS robustness check gives an opposite (negative) but also non-significant effect (p=0.171). Effects are inconsistent and not statistically significant, so the analysis does not support the hypothesis."\n}',
 1: '{\n  "answer": "No",\n  "justification": "The primary model estimates a small, non-significant effect of name femininity on log fatalities (coef=0.029, p=0.804; 95% CI for percent change: -18.3% to 29.8%). The binary female_name robustness check and damage outcome likewise show no significant effects. Confidence intervals include zero, so there is no evidence that more feminine names lead to fewer precautions (higher fatalities)."\n}'}